# Parallel Programming Lab (23AIL57)

1. **Runtime → Change runtime type → T4 GPU** (needed for programs 1, 9 and 10).
2. **Runtime → Run all**.

Each program has two cells: the first saves the code to a file (`%%writefile`), the second compiles and runs it. Programs 11 and 12 are Python and run directly.

> Free Colab has only 2 CPU cores, so OpenMP speedups stop growing after 2 threads. That is expected: speedup is limited by the number of physical cores.

## Setup
Checks the GPU and CPU count, and installs MPI (`mpicc`, `mpirun`).

In [ ]:
!nvidia-smi -L
!nproc
!pip install -q mpich

## Program 1
Write a CUDA program to perform addition of Two Arrays.

In [ ]:
%%writefile p1_array_add.cu
#include <stdio.h>
#define N 10

__global__ void add(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

int main() {
    int a[N], b[N], c[N], *da, *db, *dc, sz = N * sizeof(int);
    for (int i = 0; i < N; i++) {
        a[i] = i;
        b[i] = i * 2;
    }
    cudaMalloc((void**)&da, sz);
    cudaMalloc((void**)&db, sz);
    cudaMalloc((void**)&dc, sz);
    cudaMemcpy(da, a, sz, cudaMemcpyHostToDevice);
    cudaMemcpy(db, b, sz, cudaMemcpyHostToDevice);
    add<<<1, N>>>(da, db, dc);
    cudaMemcpy(c, dc, sz, cudaMemcpyDeviceToHost);
    for (int i = 0; i < N; i++)
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
    cudaFree(da);
    cudaFree(db);
    cudaFree(dc);
}


In [ ]:
!nvcc -arch=sm_75 p1_array_add.cu -o p1 && ./p1

## Program 2
Implement parallel matrix multiplication using OpenMP and measure execution time with varying thread counts.

In [ ]:
%%writefile p2_matmul.c
#include <stdio.h>
#include <omp.h>
#define N 500

double A[N][N], B[N][N], C[N][N];

int main() {
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            A[i][j] = 1;
            B[i][j] = 2;
        }

    double t1 = 0;
    for (int t = 1; t <= 8; t *= 2) {
        omp_set_num_threads(t);
        double start = omp_get_wtime();
        #pragma omp parallel for
        for (int i = 0; i < N; i++)
            for (int j = 0; j < N; j++) {
                double sum = 0;
                for (int k = 0; k < N; k++) sum += A[i][k] * B[k][j];
                C[i][j] = sum;
            }
        double time = omp_get_wtime() - start;
        if (t == 1) t1 = time;
        printf("Threads = %d   Time = %f s   Speedup = %.2f\n", t, time, t1 / time);
    }
    printf("C[0][0] = %.0f (expected %d)\n", C[0][0], 2 * N);
    return 0;
}


In [ ]:
!gcc -fopenmp p2_matmul.c -o p2 && ./p2

## Program 3
Develop a parallel merge sort algorithm using OpenMP and analyze speedup compared to sequential execution.

In [ ]:
%%writefile p3_mergesort.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#define N 2000000

int a[N], tmp[N], orig[N];

void merge(int l, int m, int r) {
    int i = l, j = m + 1, k = l;
    while (i <= m && j <= r) tmp[k++] = (a[i] <= a[j]) ? a[i++] : a[j++];
    while (i <= m) tmp[k++] = a[i++];
    while (j <= r) tmp[k++] = a[j++];
    for (i = l; i <= r; i++) a[i] = tmp[i];
}

void mergeSort(int l, int r) {
    if (l >= r) return;
    int m = (l + r) / 2;
    #pragma omp task if(r - l > 10000)
    mergeSort(l, m);
    #pragma omp task if(r - l > 10000)
    mergeSort(m + 1, r);
    #pragma omp taskwait
    merge(l, m, r);
}

int main() {
    for (int i = 0; i < N; i++) a[i] = orig[i] = rand();
    double start = omp_get_wtime();
    mergeSort(0, N - 1);
    double seq = omp_get_wtime() - start;
    printf("Sequential time = %f s\n\n", seq);

    for (int t = 1; t <= 8; t *= 2) {
        for (int i = 0; i < N; i++) a[i] = orig[i];
        omp_set_num_threads(t);
        start = omp_get_wtime();
        #pragma omp parallel
        #pragma omp single
        mergeSort(0, N - 1);
        double time = omp_get_wtime() - start;
        printf("Threads = %d   Time = %f s   Speedup = %.2f\n", t, time, seq / time);
    }

    for (int i = 1; i < N; i++)
        if (a[i - 1] > a[i]) { printf("NOT sorted!\n"); return 1; }
    printf("\nArray sorted correctly\n");
    return 0;
}


In [ ]:
!gcc -fopenmp p3_mergesort.c -o p3 && ./p3

## Program 4
Demonstrate how to use threadprivate and copyin directives in OpenMP to make variables local to each thread and observe the behavior of thread-private storage.

In [ ]:
%%writefile p4_threadprivate.c
#include <stdio.h>
#include <omp.h>

int x;
#pragma omp threadprivate(x)

int main() {
    omp_set_num_threads(4);
    x = 10;

    printf("Region 1 (without copyin):\n");
    #pragma omp parallel
    printf("  Thread %d: x = %d\n", omp_get_thread_num(), x);

    printf("\nRegion 2 (with copyin):\n");
    #pragma omp parallel copyin(x)
    {
        int id = omp_get_thread_num();
        printf("  Thread %d: x = %d\n", id, x);
        x += id;
    }

    printf("\nRegion 3 (persistence):\n");
    #pragma omp parallel
    printf("  Thread %d: x = %d\n", omp_get_thread_num(), x);

    printf("\nMaster thread after all regions: x = %d\n", x);
    return 0;
}


In [ ]:
!gcc -fopenmp p4_threadprivate.c -o p4 && ./p4

## Program 5
Develop an OpenMP program using various scheduling strategies (static, dynamic, guided) to observe their effect on loop performance and load balancing.

In [ ]:
%%writefile p5_schedule.c
#include <stdio.h>
#include <omp.h>
#define N 16

void work(int i) {
    for (long j = 0; j < (i + 1) * 5000000L; j++);
    printf("  Iteration %2d -> Thread %d\n", i, omp_get_thread_num());
}

int main() {
    omp_set_num_threads(4);
    double t;

    printf("STATIC:\n");
    t = omp_get_wtime();
    #pragma omp parallel for schedule(static)
    for (int i = 0; i < N; i++) work(i);
    printf("Time = %f s\n\n", omp_get_wtime() - t);

    printf("DYNAMIC:\n");
    t = omp_get_wtime();
    #pragma omp parallel for schedule(dynamic)
    for (int i = 0; i < N; i++) work(i);
    printf("Time = %f s\n\n", omp_get_wtime() - t);

    printf("GUIDED:\n");
    t = omp_get_wtime();
    #pragma omp parallel for schedule(guided)
    for (int i = 0; i < N; i++) work(i);
    printf("Time = %f s\n", omp_get_wtime() - t);
}


In [ ]:
!gcc -fopenmp p5_schedule.c -o p5 && ./p5

## Program 6
Implement a parallel prime number generation algorithm using OpenMP. Compare the performance of sequential and parallel executions by varying the number of threads and analyze execution time, speedup, and efficiency.

In [ ]:
%%writefile p6_primes.c
#include <stdio.h>
#include <omp.h>
#define N 2000000

int isPrime(int n) {
    for (int i = 2; i * i <= n; i++)
        if (n % i == 0) return 0;
    return n > 1;
}

int main() {
    int count = 0;
    double start = omp_get_wtime();
    for (int i = 2; i <= N; i++) count += isPrime(i);
    double seq = omp_get_wtime() - start;
    printf("Sequential: primes = %d, time = %f s\n\nThreads   Time(s)    Speedup   Efficiency\n", count, seq);

    for (int t = 1; t <= 8; t *= 2) {
        count = 0;
        omp_set_num_threads(t);
        start = omp_get_wtime();
        #pragma omp parallel for reduction(+:count) schedule(dynamic, 1000)
        for (int i = 2; i <= N; i++) count += isPrime(i);
        double time = omp_get_wtime() - start, sp = seq / time;
        printf("%4d    %f    %.2f      %.2f\n", t, time, sp, sp / t);
    }
    return 0;
}


In [ ]:
!gcc -fopenmp p6_primes.c -o p6 && ./p6

## Program 7
Implement a parallel summation of a large array using MPI. Use MPI_Scatter to distribute segments, compute partial sums, and use MPI_Reduce to combine results.

In [ ]:
%%writefile p7_mpi_sum.c
#include <stdio.h>
#include <mpi.h>
#define N 1000000

int a[N], part[N];

int main(int argc, char *argv[]) {
    int rank, size, local = 0, total = 0;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int chunk = N / size;
    if (rank == 0)
        for (int i = 0; i < N; i++) a[i] = 1;

    MPI_Scatter(a, chunk, MPI_INT, part, chunk, MPI_INT, 0, MPI_COMM_WORLD);
    for (int i = 0; i < chunk; i++) local += part[i];
    printf("Process %d: partial sum = %d\n", rank, local);

    MPI_Reduce(&local, &total, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);
    if (rank == 0) printf("Total sum = %d (expected %d)\n", total, N);

    MPI_Finalize();
    return 0;
}


In [ ]:
!mpicc p7_mpi_sum.c -o p7 && mpirun -np 4 ./p7

## Program 8
Create an MPI program to demonstrate both point-to-point (MPI_Send, MPI_Recv) and collective broadcast (MPI_Bcast) communication. Use a simple data structure (e.g., arrays or strings) for demonstration.

In [ ]:
%%writefile p8_mpi_comm.c
#include <stdio.h>
#include <mpi.h>

int main(int argc, char *argv[]) {
    int rank, arr[5] = {0};
    char msg[20] = "Hello from P0";

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0) {
        MPI_Send(msg, 20, MPI_CHAR, 1, 0, MPI_COMM_WORLD);
        printf("Process 0 sent: %s\n", msg);
        for (int i = 0; i < 5; i++) arr[i] = (i + 1) * 10;
    } else if (rank == 1) {
        MPI_Recv(msg, 20, MPI_CHAR, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("Process 1 received: %s\n", msg);
    }

    MPI_Bcast(arr, 5, MPI_INT, 0, MPI_COMM_WORLD);
    printf("Process %d after Bcast: %d %d %d %d %d\n", rank, arr[0], arr[1], arr[2], arr[3], arr[4]);

    MPI_Finalize();
    return 0;
}


In [ ]:
!mpicc p8_mpi_comm.c -o p8 && mpirun -np 4 ./p8

## Program 9
CUDA Program overview demonstrating kernel launch functions by varying the number of threads per block and blocks per grid and to print the GPU properties.

In [ ]:
%%writefile p9_cuda_props.cu
#include <stdio.h>

__global__ void hello() {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Block %d, Thread %d, Global ID %d\n", blockIdx.x, threadIdx.x, id);
}

int main() {
    cudaDeviceProp p;
    cudaGetDeviceProperties(&p, 0);

    printf("GPU Name             : %s\n", p.name);
    printf("Compute Capability   : %d.%d\n", p.major, p.minor);
    printf("Multiprocessors (SM) : %d\n", p.multiProcessorCount);
    printf("Warp Size            : %d\n", p.warpSize);
    printf("Max Threads/Block    : %d\n", p.maxThreadsPerBlock);

    printf("\n<<<1 block, 4 threads>>>\n");
    hello<<<1, 4>>>();
    cudaDeviceSynchronize();

    printf("\n<<<2 blocks, 4 threads>>>\n");
    hello<<<2, 4>>>();
    cudaDeviceSynchronize();

    printf("\n<<<4 blocks, 2 threads>>>\n");
    hello<<<4, 2>>>();
    cudaDeviceSynchronize();
}


In [ ]:
!nvcc -arch=sm_75 p9_cuda_props.cu -o p9 && ./p9

## Program 10
Write a CUDA program for vector addition and compare execution time with CPU-based implementation.

In [ ]:
%%writefile p10_vec_add_time.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#define N 10000000

__global__ void add(float *a, float *b, float *c) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) c[i] = a[i] + b[i];
}

int main() {
    int sz = N * sizeof(float);
    float *a = (float*)malloc(sz), *b = (float*)malloc(sz), *c = (float*)malloc(sz), *da, *db, *dc;

    for (int i = 0; i < N; i++) { a[i] = i; b[i] = i * 2; }

    clock_t start = clock();
    for (int i = 0; i < N; i++) c[i] = a[i] + b[i];
    double cpu = (double)(clock() - start) / CLOCKS_PER_SEC;

    cudaMalloc((void**)&da, sz);
    cudaMalloc((void**)&db, sz);
    cudaMalloc((void**)&dc, sz);
    cudaMemcpy(da, a, sz, cudaMemcpyHostToDevice);
    cudaMemcpy(db, b, sz, cudaMemcpyHostToDevice);

    start = clock();
    add<<<(N + 255) / 256, 256>>>(da, db, dc);
    cudaDeviceSynchronize();
    double gpu = (double)(clock() - start) / CLOCKS_PER_SEC;

    printf("CPU time = %f s\nGPU time = %f s\nSpeedup  = %.2f\n", cpu, gpu, cpu / gpu);

    cudaFree(da); cudaFree(db); cudaFree(dc);
    free(a); free(b); free(c);
}


In [ ]:
!nvcc -arch=sm_75 p10_vec_add_time.cu -o p10 && ./p10

## Program 11
Develop a Linear Regression using PyTorch to demonstrate gradient computation and basic optimization.

In [ ]:
import torch

torch.manual_seed(0)
x = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
y = 2 * x + 1

model = torch.nn.Linear(1, 1)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(1000):
    loss = loss_fn(model(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  Loss = {loss.item():.4f}  dL/dw = {model.weight.grad.item():.4f}  dL/db = {model.bias.grad.item():.4f}")

print(f"\nLearned: w = {model.weight.item():.3f}, b = {model.bias.item():.3f}  (true: w = 2, b = 1)")


## Program 12
Develop a basic feedforward neural network on random data using PyTorch.

In [ ]:
import torch

torch.manual_seed(0)
X, y = torch.randn(100, 10), torch.randn(100, 1)

model = torch.nn.Sequential(
    torch.nn.Linear(10, 16),
    torch.nn.ReLU(),
    torch.nn.Linear(16, 1)
)
print(model)

loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    loss = loss_fn(model(X), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}  Loss = {loss.item():.4f}")

print(f"Final Loss = {loss.item():.4f}")
